# Trust Detection System — Interactive Investigation + Persistent RAG (Google Colab)

This notebook combines your **existing PS-13 investigation pipeline** with a persistent, interactive evidence assistant.

### What the UI does
1. Enter a **Product URL** and **Seller ID** directly in the browser UI.
2. Click **Start Investigation** to run the actual `investigate()` pipeline from your public GitHub notebook.
3. Automatically ingest the investigation outputs into **FAISS + SQLite** stored in Google Drive.
4. Ask questions in the **Knowledge Assistant** about the current case or previous investigations.
5. See evidence IDs and source metadata with answers. The assistant must say when evidence is insufficient.

### Knowledge-base design
- There is no uploaded/predefined document collection needed for the interactive evidence RAG.
- Your investigation pipeline dynamically collects public listing/review/seller context and generates analysis outputs.
- FAISS provides semantic retrieval; SQLite applies exact seller, product URL and investigation ID filters.
- Previous investigations remain stored in Google Drive and can be searched later.

> The investigation uses public/authorized information only. Collection can be limited by marketplace access controls, availability, or API quotas. A risk signal is not proof of fraud or counterfeit activity; outputs are for human review.


## 1. Install the required packages
Run this cell once each time you start a fresh Colab runtime.

In [ ]:
# Install RAG and existing PS-13 investigation-pipeline dependencies.
!pip -q install -U sentence-transformers faiss-cpu groq gradio openai tavily-python firecrawl-py playwright networkx pandas numpy scipy scikit-learn requests beautifulsoup4 lxml python-dotenv open-clip-torch pillow opencv-python-headless
!playwright install chromium


## 2. Mount Google Drive for persistent storage
The FAISS index and SQLite database will be saved under `MyDrive/Trust_Detection_KB`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/Trust_Detection_KB")
INDEX_DIR = BASE_DIR / "faiss_index"
DB_PATH = BASE_DIR / "metadata.db"
INDEX_PATH = INDEX_DIR / "evidence.index"

BASE_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent knowledge-base folder:", BASE_DIR)

## 3. Configure the LLM and embedding model

Add your **Groq API key** when prompted. You can also save it in Colab Secrets as `GROQ_API_KEY` (key icon in the Colab sidebar).

The default Groq model below is `openai/gpt-oss-120b`. Change `GROQ_MODEL` to another model available to your Groq account. The active model list is checked through Groq's API.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") if userdata is not None else None
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Enter GROQ_API_KEY (input hidden): ").strip()

if not GROQ_API_KEY:
    raise ValueError("A Groq API key is required to use the investigation pipeline and chat assistant.")

# The existing PS-13 notebook reads Groq credentials from the environment.
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Tavily and Firecrawl are optional. Add them to Colab Secrets to enable those services.
for secret_name in ("TAVILY_API_KEY", "FIRECRAWL_API_KEY"):
    try:
        secret_value = os.getenv(secret_name) or (userdata.get(secret_name) if userdata is not None else None)
    except Exception:
        secret_value = os.getenv(secret_name)
    if secret_value:
        os.environ[secret_name] = secret_value

from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "openai/gpt-oss-120b"  # Change to a model enabled for your Groq account.

try:
    active_models = {m.id for m in groq_client.models.list().data if getattr(m, "active", True)}
    if GROQ_MODEL in active_models:
        print(f"Groq model is available: {GROQ_MODEL}")
    else:
        print("Configured model was not listed as active for this API key.")
        print("Some available model IDs:", sorted(active_models)[:20])
        print("Change GROQ_MODEL above to an ID from this list, then rerun this cell.")
except Exception as exc:
    print("Could not verify the model list. The chat call will test the configured model.")
    print("Model-list check:", str(exc)[:300])

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 6
CHUNK_SIZE = 1100
CHUNK_OVERLAP = 160
print("Embedding model:", EMBEDDING_MODEL)
print("Tavily:", "configured" if os.getenv("TAVILY_API_KEY") else "not configured (optional)")
print("Firecrawl:", "configured" if os.getenv("FIRECRAWL_API_KEY") else "not configured (Playwright fallback may be used)")


## 4. Load the embedding model and initialize SQLite + FAISS
SQLite stores exact identifiers and evidence metadata. FAISS stores normalized embeddings for semantic retrieval.

In [ ]:
import json
import re
import sqlite3
import uuid
from datetime import datetime, timezone
from typing import Any

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
EMBEDDING_DIM = embedder.get_sentence_embedding_dimension()


from contextlib import contextmanager

@contextmanager
def db_connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def initialize_database():
    with db_connect() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS evidence_chunks (
                row_id INTEGER PRIMARY KEY AUTOINCREMENT,
                evidence_id TEXT NOT NULL,
                chunk_no INTEGER NOT NULL,
                investigation_id TEXT,
                seller_id TEXT,
                product_id TEXT,
                product_url TEXT,
                agent TEXT,
                evidence_type TEXT,
                source TEXT,
                source_url TEXT,
                content TEXT NOT NULL,
                metadata_json TEXT,
                created_at TEXT NOT NULL,
                vector_id INTEGER UNIQUE,
                UNIQUE(evidence_id, chunk_no)
            )
        """)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_evidence_seller ON evidence_chunks(seller_id)")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_evidence_product ON evidence_chunks(product_url)")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_evidence_investigation ON evidence_chunks(investigation_id)")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_evidence_id ON evidence_chunks(evidence_id)")
    print("SQLite evidence table is ready.")


initialize_database()


def rebuild_faiss_index():
    """Rebuild FAISS from SQLite if the saved index is missing or out of sync."""
    global faiss_index
    with db_connect() as conn:
        rows = conn.execute(
            "SELECT row_id, content FROM evidence_chunks ORDER BY row_id"
        ).fetchall()

    faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)  # cosine similarity after normalization
    if rows:
        texts = [row["content"] for row in rows]
        vectors = embedder.encode(
            texts, convert_to_numpy=True, normalize_embeddings=True,
            show_progress_bar=True
        ).astype("float32")
        faiss_index.add(vectors)
        with db_connect() as conn:
            for vector_id, row in enumerate(rows):
                conn.execute(
                    "UPDATE evidence_chunks SET vector_id=? WHERE row_id=?",
                    (vector_id, row["row_id"])
                )
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    faiss.write_index(faiss_index, str(INDEX_PATH))
    print(f"FAISS index rebuilt with {faiss_index.ntotal} evidence chunks.")


def load_faiss_index():
    global faiss_index
    if not INDEX_PATH.exists():
        rebuild_faiss_index()
        return

    try:
        saved_index = faiss.read_index(str(INDEX_PATH))
        with db_connect() as conn:
            vector_rows = conn.execute(
                "SELECT vector_id FROM evidence_chunks ORDER BY vector_id"
            ).fetchall()
        vector_ids = [r["vector_id"] for r in vector_rows]
        is_aligned = (
            saved_index.d == EMBEDDING_DIM
            and saved_index.ntotal == len(vector_ids)
            and vector_ids == list(range(len(vector_ids)))
        )
        if not is_aligned:
            rebuild_faiss_index()
        else:
            faiss_index = saved_index
            print(f"Loaded persistent FAISS index: {faiss_index.ntotal} chunks.")
    except Exception as exc:
        print("Saved index could not be loaded; rebuilding from SQLite:", str(exc)[:250])
        rebuild_faiss_index()


load_faiss_index()

## 5. Load your existing Trust Detection agents

This cell loads the real agent implementation from your public repository `786-sys/AAIEA_MINI_PROJECT`, then exposes its `investigate(product_url, seller_id, ...)` function to the UI. The source notebook is fetched from `project/AAI_MINI_PROJECT.ipynb`; only the API/configuration, collector, analysis, evidence, risk and investigation cells are executed. The original Streamlit launcher and sample product runs are not executed.

The pipeline's Sentence Transformer is pointed at the same `embedder` already loaded for RAG to avoid loading a duplicate text-embedding model.


In [ ]:
import requests

PIPELINE_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/786-sys/AAIEA_MINI_PROJECT/main/"
    "project/AAI_MINI_PROJECT.ipynb"
)
PIPELINE_CELL_IDS = [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 36]

response = requests.get(PIPELINE_NOTEBOOK_URL, timeout=60)
response.raise_for_status()
source_notebook = response.json()

if len(source_notebook.get("cells", [])) <= max(PIPELINE_CELL_IDS):
    raise RuntimeError("The source Trust Detection notebook has changed; expected pipeline cells were not found.")

for cell_id in PIPELINE_CELL_IDS:
    cell = source_notebook["cells"][cell_id]
    if cell.get("cell_type") != "code":
        raise TypeError(f"Expected code cell {cell_id} in the source Trust Detection notebook.")
    cell_source = "".join(cell.get("source", []))

    # Reuse the RAG embedding model already loaded above instead of downloading a second copy.
    if cell_id == 6:
        cell_source = cell_source.replace(
            "sentence_model = SentenceTransformer('all-MiniLM-L6-v2')",
            "sentence_model = embedder"
        )

    # Retain parsed public review text in the investigation result so chat can cite review examples.
    if cell_id == 30:
        nl = chr(10)
        cell_source = cell_source.replace(
            "    review_doc = collect_reviews(product_url)",
            "    review_doc = collect_reviews(product_url)" + nl + "    state.product_data = product_snapshot",
        )
        cell_source = cell_source.replace(
            "    else:" + nl + "        review_doc['reviews'] = []" + nl + "    record('Review collection', {",
            "    else:" + nl + "        review_doc['reviews'] = []" + nl + "    state.review_data = review_doc" + nl + "    record('Review collection', {",
        )
        cell_source = cell_source.replace(
            "        'review_analysis': statistical,",
            "        'review_analysis': statistical," + nl
            + "        'collected_reviews': review_doc.get('reviews', [])," + nl
            + "        'review_collection_metadata': {k: v for k, v in review_doc.items() if k not in ('raw_content', 'reviews')},",
        )


    exec(compile(cell_source, f"trust_detection_source_cell_{cell_id}", "exec"), globals())

# The corrected orchestrator parser is in source cell 36 and is executed last by the list above.
print("Existing Trust Detection pipeline loaded successfully.")
print("Investigation function available:", callable(globals().get("investigate")))


## 6. Evidence ingestion functions

The input can be a dictionary, a list of structured findings, or plain text returned by an existing agent. Each agent output is split into chunks, embedded, and stored with its seller/product/investigation metadata.

**Important integration point:** after your existing agents finish, pass their outputs to `ingest_agent_outputs(...)`. No static document collection is required.

In [ ]:
def flatten_payload(value: Any, prefix: str = "") -> str:
    """Convert nested agent output into readable text without discarding field names."""
    lines = []
    if isinstance(value, dict):
        for key, item in value.items():
            key_path = f"{prefix}.{key}" if prefix else str(key)
            if item is None:
                continue
            if isinstance(item, (dict, list)):
                nested = flatten_payload(item, key_path)
                if nested:
                    lines.append(nested)
            else:
                lines.append(f"{key_path}: {item}")
    elif isinstance(value, list):
        for i, item in enumerate(value):
            nested = flatten_payload(item, f"{prefix}[{i}]")
            if nested:
                lines.append(nested)
    else:
        text_value = str(value).strip()
        if text_value:
            lines.append(f"{prefix}: {text_value}" if prefix else text_value)
    return "\n".join(lines).strip()


def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start + int(chunk_size * 0.65):
                end = boundary
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return [c for c in chunks if c]


def pick_value(item: dict, *keys, default=None):
    for key in keys:
        value = item.get(key)
        if value not in (None, ""):
            return value
    return default


def ingest_agent_outputs(
    agent_outputs: dict,
    seller_id: str = "",
    product_url: str = "",
    product_id: str = "",
    investigation_id: str = "",
):
    """Ingest {agent_name: output} from the existing Trust Detection agents."""
    global faiss_index
    if not isinstance(agent_outputs, dict):
        raise TypeError("agent_outputs must be a dictionary: {agent_name: output}")
    if not agent_outputs:
        print("No agent outputs supplied. Populate agent_outputs in the next cell first.")
        return 0

    investigation_id = investigation_id or f"INV-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:6]}"
    added_chunks = 0
    skipped_chunks = 0

    for agent_name, payload in agent_outputs.items():
        if payload is None:
            continue
        items = payload if isinstance(payload, list) else [payload]
        for item_number, item in enumerate(items):
            item_dict = item if isinstance(item, dict) else {"agent_output": str(item)}
            content = flatten_payload(item_dict)
            if not content:
                continue

            supplied_id = pick_value(item_dict, "evidence_id", "finding_id", "id")
            base_id = str(supplied_id) if supplied_id else f"{agent_name}-{item_number}-{uuid.uuid4().hex[:8]}"
            evidence_id = f"{investigation_id}:{agent_name}:{base_id}"
            row_seller = str(pick_value(item_dict, "seller_id", default=seller_id) or "")
            row_product_url = str(pick_value(item_dict, "product_url", "url", default=product_url) or "")
            row_product_id = str(pick_value(item_dict, "product_id", default=product_id) or "")
            evidence_type = str(pick_value(item_dict, "evidence_type", "type", "category", default="agent_finding"))
            source = str(pick_value(item_dict, "source", "source_name", default=agent_name))
            source_url = str(pick_value(item_dict, "source_url", "evidence_url", default="") or "")
            created_at = str(pick_value(item_dict, "timestamp", "created_at", default=datetime.now(timezone.utc).isoformat()))
            metadata_json = json.dumps(item_dict, ensure_ascii=False, default=str)

            for chunk_no, chunk in enumerate(split_into_chunks(content)):
                with db_connect() as conn:
                    exists = conn.execute(
                        "SELECT 1 FROM evidence_chunks WHERE evidence_id=? AND chunk_no=?",
                        (evidence_id, chunk_no)
                    ).fetchone()
                if exists:
                    skipped_chunks += 1
                    continue

                vector = embedder.encode(
                    [chunk], convert_to_numpy=True, normalize_embeddings=True
                ).astype("float32")
                vector_id = int(faiss_index.ntotal)
                with db_connect() as conn:
                    conn.execute("""
                        INSERT INTO evidence_chunks (
                            evidence_id, chunk_no, investigation_id, seller_id,
                            product_id, product_url, agent, evidence_type, source,
                            source_url, content, metadata_json, created_at, vector_id
                        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """, (
                        evidence_id, chunk_no, investigation_id, row_seller,
                        row_product_id, row_product_url, str(agent_name), evidence_type,
                        source, source_url, chunk, metadata_json, created_at, vector_id
                    ))
                faiss_index.add(vector)
                added_chunks += 1

    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    faiss.write_index(faiss_index, str(INDEX_PATH))
    print(f"Investigation ID: {investigation_id}")
    print(f"New evidence chunks added: {added_chunks}; duplicates skipped: {skipped_chunks}")
    print(f"Total chunks in persistent knowledge base: {faiss_index.ntotal}")
    return {
        "investigation_id": investigation_id,
        "added_chunks": added_chunks,
        "total_chunks": int(faiss_index.ntotal),
    }


def ingest_evidence_records(records: list, **scope):
    """Optional: ingest a flat list of evidence dictionaries."""
    grouped = {}
    for i, record in enumerate(records):
        if not isinstance(record, dict):
            record = {"evidence": str(record)}
        agent = str(record.get("agent", "manual_evidence"))
        grouped.setdefault(agent, []).append(record)
    return ingest_agent_outputs(grouped, **scope)


def knowledge_base_stats():
    with db_connect() as conn:
        total = conn.execute("SELECT COUNT(*) FROM evidence_chunks").fetchone()[0]
        investigations = conn.execute("SELECT COUNT(DISTINCT investigation_id) FROM evidence_chunks").fetchone()[0]
        sellers = conn.execute("SELECT COUNT(DISTINCT seller_id) FROM evidence_chunks WHERE seller_id <> ''").fetchone()[0]
        agents = conn.execute("SELECT DISTINCT agent FROM evidence_chunks ORDER BY agent").fetchall()
    return {
        "evidence_chunks": total,
        "investigations": investigations,
        "sellers": sellers,
        "agents": [r[0] for r in agents],
    }

print("Evidence ingestion functions are ready.")

## 7. Investigation data is now entered in the UI

You no longer need to paste Product URL, Seller ID or agent-output variables into this notebook cell. The **New Investigation** form in the final UI calls the real `investigate()` function, then converts its structured results into evidence records and automatically stores them in FAISS + SQLite.

Previous cases are retained in Google Drive. Use the filters in the Knowledge Assistant to query one seller, one product, one investigation, or the full history.


In [ ]:
# Product URL and Seller ID are now entered in the browser UI below.
# No static sample investigation is run from this notebook cell.
print("Use the New Investigation form in the UI to start a case.")


## 8. Hybrid retrieval: metadata filters + semantic FAISS search
Seller ID, product URL and investigation ID are exact filters in SQLite. FAISS then ranks semantically relevant evidence within the selected scope.

In [ ]:
def retrieve_evidence(
    question: str,
    seller_id: str = "",
    product_url: str = "",
    investigation_id: str = "",
    top_k: int = TOP_K,
):
    if not question.strip() or faiss_index.ntotal == 0:
        return []

    query_vector = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, vector_ids = faiss_index.search(query_vector, faiss_index.ntotal)

    with db_connect() as conn:
        rows = conn.execute("SELECT * FROM evidence_chunks").fetchall()
    by_vector_id = {int(row["vector_id"]): dict(row) for row in rows if row["vector_id"] is not None}

    results = []
    for score, vector_id in zip(scores[0], vector_ids[0]):
        if vector_id < 0:
            continue
        row = by_vector_id.get(int(vector_id))
        if not row:
            continue
        if seller_id and row["seller_id"] != seller_id:
            continue
        if product_url and row["product_url"] != product_url:
            continue
        if investigation_id and row["investigation_id"] != investigation_id:
            continue
        row["similarity"] = float(score)
        results.append(row)
        if len(results) >= top_k:
            break
    return results


print("Retriever ready.")

## 9. Grounded RAG question-answering

The LLM receives only the question, recent chat context, and retrieved evidence. It must not assume that an anomaly is proof of fraud, and it must say when the available evidence does not answer the question.

In [ ]:
SYSTEM_PROMPT = """You are the evidence assistant for a Trust Detection System.
Answer questions using ONLY the retrieved investigation evidence supplied in the prompt.
Rules:
1. Do not invent review counts, dates, seller history, URLs, risk scores, model results, or evidence.
2. If the evidence does not answer the question, clearly say: 'The current knowledge base does not contain sufficient evidence to answer this.'
3. Distinguish observed facts, agent-generated signals, and interpretation.
4. A suspicious signal is not proof of fraud or counterfeit activity.
5. Cite evidence IDs exactly as provided, for example [INV-...:statistical_agent:...].
6. Do not claim that an external source was verified unless its source details are present in the retrieved evidence.
7. Be direct and concise. Mention conflicts or missing information when relevant."""


def history_to_text(history, max_messages=6):
    lines = []
    for message in (history or [])[-max_messages:]:
        if not isinstance(message, dict):
            continue
        role = message.get("role", "user")
        content = message.get("content", "")
        if isinstance(content, list):
            text_parts = []
            for part in content:
                if isinstance(part, dict) and part.get("type") == "text":
                    text_parts.append(part.get("text", ""))
                elif isinstance(part, str):
                    text_parts.append(part)
            content = " ".join(text_parts)
        lines.append(f"{role}: {content}")
    return "\n".join(lines)


def ask_rag(
    question: str,
    seller_id: str = "",
    product_url: str = "",
    investigation_id: str = "",
    history=None,
    top_k: int = TOP_K,
):
    if not question or not question.strip():
        return "Please enter a question."

    conversation_context = history_to_text(history)
    retrieval_query = (
        f"{conversation_context}\nCurrent question: {question}"
        if conversation_context else question
    )
    matches = retrieve_evidence(
        retrieval_query,
        seller_id=seller_id.strip(),
        product_url=product_url.strip(),
        investigation_id=investigation_id.strip(),
        top_k=top_k,
    )
    if not matches:
        return (
            "I could not find matching evidence in the selected knowledge-base scope. "
            "First run your investigation agents and ingest their outputs. "
            "Also check that the Seller ID, Product URL and Investigation ID filters match the stored values."
        )

    evidence_blocks = []
    for row in matches:
        evidence_blocks.append(
            f"[EVIDENCE_ID: {row['evidence_id']}]\n"
            f"Agent: {row['agent']} | Type: {row['evidence_type']} | Source: {row['source']}\n"
            f"Seller: {row['seller_id']} | Product URL: {row['product_url']} | Investigation: {row['investigation_id']}\n"
            f"Source URL: {row['source_url']} | Recorded at: {row['created_at']}\n"
            f"Retrieved similarity: {row['similarity']:.3f}\n"
            f"Evidence content: {row['content']}"
        )
    evidence_context = "\n\n---\n\n".join(evidence_blocks)
    user_prompt = f"""Recent conversation (for follow-up context only):
{conversation_context or '(No earlier messages)'}

Current question:
{question}

Selected filters:
Seller ID: {seller_id or 'All sellers'}
Product URL: {product_url or 'All products'}
Investigation ID: {investigation_id or 'All investigations'}

Retrieved evidence:
{evidence_context}

Answer using only the retrieved evidence. Cite the exact evidence IDs that support your answer. If it is insufficient, say so."""

    try:
        completion = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.1,
            max_completion_tokens=1200,
        )
        answer = completion.choices[0].message.content or "The model returned an empty response."
    except Exception as exc:
        return f"Groq request failed: {exc}\n\nCheck GROQ_API_KEY and GROQ_MODEL."

    # Attach a deterministic source list from the rows actually retrieved.
    seen = set()
    source_lines = []
    for row in matches:
        key = (row["evidence_id"], row["agent"], row["source"], row["source_url"])
        if key in seen:
            continue
        seen.add(key)
        source_line = f"- `{row['evidence_id']}` — {row['agent']} / {row['source']}"
        if row["source_url"]:
            source_line += f" — {row['source_url']}"
        source_lines.append(source_line)

    return answer + "\n\n**Retrieved evidence sources**\n" + "\n".join(source_lines)


print("Grounded RAG question-answering is ready.")

## 10. Inspect the knowledge base

Run this at any time to check how many investigations, sellers and evidence chunks have been stored.

In [ ]:
from pprint import pprint
pprint(knowledge_base_stats())

## 11. Launch the combined investigation + RAG UI

The interface has two sequential sections on one page:

1. **New Investigation:** Product URL + Seller ID + Start Investigation. It runs the existing PS-13 pipeline and automatically ingests all structured outputs.
2. **Knowledge Assistant:** chat questions with seller/product/investigation filters. After an investigation, the filters auto-fill to the current case. Clear the investigation filter to compare earlier cases.

The Gradio share link is public while the Colab runtime is active. Use public/authorized data only.


In [ ]:
import gradio as gr
from urllib.parse import urlparse


def _build_rag_payload(investigation_result):
    """Convert the real PS-13 pipeline result into searchable evidence payloads."""
    return {
        "product_snapshot": investigation_result.get("product_snapshot", {}),
        "statistical_agent": investigation_result.get("review_analysis", {}),
        "collected_reviews": investigation_result.get("collected_reviews", []),
        "review_collection_metadata": investigation_result.get("review_collection_metadata", {}),
        "linguistic_agent": investigation_result.get("linguistic_analysis", {}),
        "reviewer_network_agent": investigation_result.get("network_analysis", {}),
        "claim_verification_agent": investigation_result.get("claim_verification", {}),
        "visual_similarity_agent": investigation_result.get("visual_analysis", {}),
        "seller_risk_agent": investigation_result.get("seller_risk", {}),
        "evidence_records": investigation_result.get("evidence", []),
        "compliance_mapping": investigation_result.get("compliance_mapping", {}),
        "risk_summary": investigation_result.get("risk_summary", {}),
        "final_report": investigation_result.get("report", {}),
        "pipeline_trace": investigation_result.get("pipeline_trace", []),
    }


def run_investigation_ui(product_url, seller_id, progress=gr.Progress(track_tqdm=False)):
    """Run the existing PS-13 agents, then persist their outputs in the RAG knowledge base."""
    product_url = (product_url or "").strip()
    seller_id = (seller_id or "").strip()

    parsed = urlparse(product_url)
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        return "**Please enter a valid public product URL beginning with https:// or http://.**", {}, [], "", "", ""
    if not seller_id:
        return "**Please enter the Seller ID shown on the listing.**", {}, [], "", "", ""
    if not callable(globals().get("investigate")):
        return "**The Trust Detection investigation pipeline is not loaded. Rerun the pipeline-loader cell above.**", {}, [], "", "", ""

    stage_counter = [0]
    stage_names = []

    def on_stage(stage_name, payload):
        stage_counter[0] += 1
        stage_names.append(str(stage_name))
        # Keep UI progress informative without displaying large raw payloads in the progress bar.
        progress((min(stage_counter[0], 14), 14), desc=str(stage_name)[:70])

    try:
        investigation_result = investigate(
            product_url=product_url,
            seller_id=seller_id,
            verbose=False,
            progress_callback=on_stage,
        )

        rag_payload = _build_rag_payload(investigation_result)
        ingestion = ingest_agent_outputs(
            rag_payload,
            seller_id=seller_id,
            product_url=product_url,
            product_id=str((investigation_result.get("product_snapshot") or {}).get("product_id") or ""),
        )

        report = investigation_result.get("report", {}) or {}
        current_investigation_id = ingestion.get("investigation_id", "")
        status_label = "completed" if investigation_result.get("status") == "ok" else "completed with limited evidence"
        status_md = (
            f"### Investigation {status_label}\n\n"
            f"- **Seller ID:** `{seller_id}`\n"
            f"- **Risk score:** {report.get('risk_score', 'N/A')} / 100\n"
            f"- **Risk level:** {report.get('risk_level', 'Unavailable')}\n"
            f"- **Evidence items from pipeline:** {len(investigation_result.get('evidence', []))}\n"
            f"- **RAG chunks stored:** {ingestion.get('added_chunks', 0)}\n"
            f"- **Investigation ID:** `{current_investigation_id}`\n\n"
            "The knowledge assistant filters are set to this case. To compare all previous cases for this seller, keep Seller ID and clear both Product URL and Investigation ID filters."
        )
        return (
            status_md,
            report,
            investigation_result.get("evidence", []),
            seller_id,
            product_url,
            current_investigation_id,
        )
    except Exception as exc:
        return f"### Investigation failed\n\n`{type(exc).__name__}: {str(exc)}`", {}, [], seller_id, product_url, ""


def chat_submit(message, history, seller_filter, product_filter, investigation_filter):
    history = history or []
    message = (message or "").strip()
    if not message:
        return history, ""

    answer = ask_rag(
        question=message,
        seller_id=(seller_filter or "").strip(),
        product_url=(product_filter or "").strip(),
        investigation_id=(investigation_filter or "").strip(),
        history=history,
    )
    updated_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return updated_history, ""


def clear_chat():
    return [], ""


with gr.Blocks(title="Trust Detection — Investigation + Evidence Assistant", theme=gr.themes.Soft()) as app:
    gr.Markdown(
        "# Trust Detection — Investigation & Knowledge Assistant\n"
        "Enter a Product URL and Seller ID to investigate. The findings are saved automatically, "
        "then you can ask questions grounded in current and historical evidence."
    )

    with gr.Group():
        gr.Markdown("## A. New Investigation")
        with gr.Row():
            product_url_input = gr.Textbox(
                label="Product URL",
                placeholder="https://marketplace.example/product/123",
                scale=3,
            )
            seller_id_input = gr.Textbox(
                label="Seller ID",
                placeholder="Enter the seller identifier shown on the listing",
                scale=2,
            )
        start_button = gr.Button("Start Investigation", variant="primary", size="lg")
        investigation_status = gr.Markdown("Enter the Product URL and Seller ID, then click **Start Investigation**.")
        with gr.Accordion("Investigation output", open=False):
            report_output = gr.JSON(label="Investigation report")
            evidence_output = gr.JSON(label="Evidence records")

    gr.Markdown("---")
    gr.Markdown("## B. Interactive Knowledge Assistant")
    gr.Markdown(
        "Ask questions about the investigation you just ran or search earlier cases. "
        "After a new investigation, the filters below are automatically filled for that case. To compare all previous cases for one seller, keep Seller ID and clear Product URL and Investigation ID."
    )

    with gr.Accordion("Search filters (optional)", open=True):
        with gr.Row():
            seller_filter = gr.Textbox(label="Seller ID filter", placeholder="Leave blank to search all sellers")
            product_filter = gr.Textbox(label="Product URL filter", placeholder="Leave blank to search all products")
            investigation_filter = gr.Textbox(label="Investigation ID filter", placeholder="Leave blank to search all investigations")

    chatbot = gr.Chatbot(label="Evidence Assistant", type="messages", height=440)
    with gr.Row():
        question_input = gr.Textbox(
            label="Your question",
            placeholder="e.g. Why was this seller flagged? Show supporting evidence.",
            lines=2,
            scale=8,
        )
        send_button = gr.Button("Send", variant="primary", scale=1)
        clear_button = gr.Button("Clear chat", scale=1)

    gr.Examples(
        examples=[
            ["Why was this seller flagged? Show supporting evidence."],
            ["Which anomalies were reported by the agents?"],
            ["What evidence is missing before we can make a conclusion?"],
            ["Compare this seller's findings across previous investigations."],
            ["Summarize the product, review, and seller risk signals."],
        ],
        inputs=question_input,
        label="Suggested questions",
    )

    start_button.click(
        fn=run_investigation_ui,
        inputs=[product_url_input, seller_id_input],
        outputs=[investigation_status, report_output, evidence_output, seller_filter, product_filter, investigation_filter],
    )
    send_button.click(
        fn=chat_submit,
        inputs=[question_input, chatbot, seller_filter, product_filter, investigation_filter],
        outputs=[chatbot, question_input],
    )
    question_input.submit(
        fn=chat_submit,
        inputs=[question_input, chatbot, seller_filter, product_filter, investigation_filter],
        outputs=[chatbot, question_input],
    )
    clear_button.click(fn=clear_chat, inputs=[], outputs=[chatbot, question_input])

# Serialize investigations to avoid concurrent mutations of the existing pipeline's shared state.evidence object.
app.queue(default_concurrency_limit=1)
app.launch(share=True, debug=True)


## 12. Adding evidence from a separate/manual agent run

Run this cell after your existing agents finish a new investigation. It adds only new evidence; previous investigations remain in the knowledge base.

```python
next_agent_outputs = {
    "statistical_agent": statistical_result,
    "linguistic_agent": linguistic_result,
    "reviewer_network_agent": reviewer_network_result,
    "claim_verification_agent": claim_verification_result,
    "visual_similarity_agent": visual_result,
    "seller_risk_agent": seller_risk_result,
}

ingestion_result = ingest_agent_outputs(
    next_agent_outputs,
    seller_id=current_seller_id,
    product_url=current_product_url,
)
print("Stored investigation:", ingestion_result["investigation_id"])
```

Replace the example variable names with the actual variables from your project. You can also ingest just the agents that ran successfully. The knowledge base preserves seller IDs, product URLs, investigation IDs, source labels and timestamps for historical retrieval.

## 13. Implementation notes

- The New Investigation button invokes the existing `investigate()` pipeline from your GitHub notebook; it is not a dummy callback.
- The RAG layer ingests the resulting product snapshot, statistical/linguistic/network/claim/visual/seller analyses, evidence records, compliance mapping, risk summary, final report and pipeline trace.
- SQLite stores exact seller/product/investigation metadata; FAISS stores semantic vectors.
- The notebook retrieves the original agent code from your public GitHub repository at runtime. Keep that repository public or change `PIPELINE_NOTEBOOK_URL` to an accessible raw notebook URL.
- Product pages may restrict access or expose incomplete information. The UI reports limitations instead of treating unavailable data as proof of low risk.
- The notebook does not automatically enforce marketplace actions.

### References
- [Trust Detection source notebook](https://github.com/786-sys/AAIEA_MINI_PROJECT/blob/main/project/AAI_MINI_PROJECT.ipynb)
- [FAISS](https://github.com/facebookresearch/faiss)
- [Sentence Transformers documentation](https://sbert.net/)
- [Gradio Blocks](https://www.gradio.app/docs/gradio/blocks)
- [Groq supported models](https://console.groq.com/docs/models)


## Run order and first use

1. Run cells from top to bottom through **Load your existing Trust Detection agents**.
2. Complete Groq setup; add `TAVILY_API_KEY` and `FIRECRAWL_API_KEY` in Colab Secrets for fuller collection coverage (optional but recommended).
3. Run the final UI cell.
4. In the browser interface, enter Product URL + Seller ID and click **Start Investigation**.
5. Wait for the investigation to finish. Its outputs are automatically saved to the persistent knowledge base.
6. Ask questions in the Knowledge Assistant. The current case filters are pre-filled; clear the Investigation ID to compare previous investigations.

The notebook loads the agent pipeline from the public GitHub path shown in the pipeline-loader cell. It does not run the old Streamlit launcher or the sample product test cells from that source notebook.
